# Customer operations

Refunds, disputes, a short contract, an invoice matched to a purchase order, and two sales notes. Days since purchase are already stored on the order. Jev is not asked to subtract dates.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 59. Is this refund inside the policy?

Two orders: A-201 is 10 days old with broken poles. A-330 is 45 days old. The policy window is 30 days.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    policy = read_text("refund-policy.md")
    questions = {
        "refund_requested": Noul(instructions="Does `message` explicitly ask for money back or a credit?"),
        "policy_covers": Noul(instructions="Does `policy` cover the situation in `message`, given `order`?"),
        "within_window": Noul(instructions="Is `order.days_since_purchase_text` inside the window stated in `policy`?"),
    }
    for ticket_id, order_id in (("T-201", "A-201"), ("T-330", "A-330")):
        record = order(order_id)
        record["days_since_purchase_text"] = "%s days ago" % record["days_since_purchase"]
        response = ask(
            {"message": ticket(ticket_id)["body"], "order": record, "policy": policy},
            questions,
        )
        show(response)
        if response.nouls["refund_requested"].noul < 0.55:
            route = "not_a_refund"
        elif min(response.nouls["policy_covers"].noul, response.nouls["within_window"].noul) > 0.65:
            route = "eligible"
        else:
            route = "agent_review"
        print(order_id, "->", route)


**What you should see.** A-201 should be eligible. A-330 is outside 30 days, so it should go to review rather than auto-approve.


## 60. Code a cardholder dispute

Pick a reason code, and separately ask whether the shop's evidence answers the claim.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    item = load_json("work.json")["dispute"]
    response = ask(
        item,
        {
            "reason_code": Choice(
                instructions="Which dispute reason best matches `cardholder_statement`?",
                criteria=item["reason_codes"],
            ),
            "evidence_sufficient": Noul(instructions="Does `merchant_evidence` directly answer the claim in `cardholder_statement`?"),
        },
    )
    show(response)
    code = response.choices["reason_code"]
    label = code.choice if code.confidence > 0.6 else "manual"
    print({"code": label, "auto_respond": response.nouls["evidence_sufficient"].noul > 0.7})


**What you should see.** The two $49 charges should code as duplicate. The evidence quotes those two lines, so an automatic response is reasonable if confidence is high.


## 61. Pick a contract clause from the candidates code found

Python splits the contract on numbered clauses. Jev picks the index. The text that comes back is the original span.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    contract = read_text("contract.txt")
    chunks = [chunk.strip() for chunk in re.split(r"\n\s*\d+\.\s+", contract) if len(chunk.strip()) > 40]
    print("chunks:", len(chunks))
    response = ask(
        {"contract": contract},
        {
            "which": Choice(
                instructions="Which candidate is the returns clause?",
                criteria={str(i): chunk[:180] for i, chunk in enumerate(chunks)},
            ),
            "exists": Noul(instructions="Does `contract` contain a returns clause?"),
        },
    )
    show(response)
    if response.nouls["exists"].noul < 0.5 or response.choices["which"].confidence < 0.5:
        print("route: not found")
    else:
        print(chunks[int(response.choices["which"].choice)])


**What you should see.** The clause about wholesale returns and 30 days should be the one printed.


## 62. Match invoice lines to purchase-order lines

Jev only decides whether two descriptions are the same item. The dollar difference is subtracted in Python.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    invoice = load_json("invoices.json")
    left = invoice["invoice_lines"]
    right = invoice["po_lines"]
    questions = {
        "same_%s_%s" % (i, j): Noul(
            instructions="Do invoice[%s].description and po[%s].description describe the same item?" % (i, j)
        )
        for i in range(len(left))
        for j in range(len(right))
    }
    response = ask({"invoice": left, "po": right}, questions)
    show(response)
    for i, line in enumerate(left):
        best_j, best_p = max(
            ((j, response.nouls["same_%s_%s" % (i, j)].noul) for j in range(len(right))),
            key=lambda item: item[1],
        )
        if best_p > 0.65:
            gap = abs(line["amount"] - right[best_j]["amount"])
            print(i, "->", best_j, "gap", gap)
        else:
            print(i, "->", None)


**What you should see.** Both backpack lines should match a PO line, and the dollar gap should be 0.


## 63. Lead score from four yes/no questions

Budget, timing, authority, and a clear problem. The weights are in the cell.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "budget_mentioned": Noul(instructions="Does `notes` mention a budget or a willingness to pay?"),
        "timeline_stated": Noul(instructions="Does `notes` say when they want to start or decide?"),
        "decision_maker": Noul(instructions="Does `notes` indicate this person can approve the purchase?"),
        "pain_clear": Noul(instructions="Does `notes` describe a concrete problem?"),
    }
    for lead in load_json("leads.json"):
        response = ask(lead, questions)
        show(response)
        score = (
            0.3 * response.nouls["budget_mentioned"].noul
            + 0.25 * response.nouls["timeline_stated"].noul
            + 0.25 * response.nouls["decision_maker"].noul
            + 0.2 * response.nouls["pain_clear"].noul
        )
        if score > 0.65:
            band = "hot"
        elif score > 0.4:
            band = "warm"
        else:
            band = "nurture"
        print(lead["name"], round(score, 2), band)


**What you should see.** Camp Cedar, with a budget, a month, and broken poles, should be hot. Alex, who is browsing, should be nurture.
